# GRPO++ : from vanilla GRPO to the tricks that make RL actually work

A **from-scratch, transparent** implementation of Group Relative Policy Optimization and the
modern fixes on top of it, built so every concept in Cameron Wolfe's *GRPO++* overview is
**visible and ablatable** rather than hidden behind a `Trainer`. We train a small open Qwen
model with **verifiable rewards (RLVR)** on GSM8K math.

Everything below is implemented in plain `transformers` + `torch` (no `trl` for the core loop)
so you can read the actual math: group-relative advantage, the clipped surrogate, the loss
aggregation choices, and the variant objectives.

**What's covered (paper → knob you can flip):**

| # | Idea | Paper |
|---|------|-------|
| 1 | Group-relative advantage, no critic | GRPO [13] |
| 2 | Clip-higher (decoupled ε) | DAPO [1] |
| 3 | Token-level loss (kills length bias) | DAPO [1] |
| 4 | Dynamic sampling (drop zero-advantage groups) | DAPO [1] |
| 5 | Overlong masking / soft length penalty | DAPO [1] |
| 6 | Drop std-normalization + constant-length norm | Dr. GRPO [3] |
| 7 | No-KL training | common |
| 8 | Sequence-level importance ratio | GSPO [6] |
| 9 | Soft token clipping (keep "fork" tokens) | CISPO [8] |
| 10 | Truncated Importance Sampling (sampler≠learner) | TIS [4] |
| 11 | Health metrics: reward / length / entropy / clip-frac | DAPO [1] |

> **Runtime answer up front:** the default config (Qwen2.5-0.5B-Instruct, short completions)
> runs on a **free Google Colab T4**. RunPod is only needed once you scale to 1.5B–7B,
> full fine-tuning, or vLLM-accelerated rollouts. See the final cell for exact sizing.
>
> For a **fast smoke test** (~10 min) set `cfg.num_steps=30`, `cfg.n_train=200`,
> `cfg.dynamic_sampling=False`. For a real run leave defaults (you'll see reward + format
> move within ~50–100 steps).

In [ ]:
# === Dependencies (run once in Colab) ============================================
# transformers >= 4.45 is needed for the logits_to_keep memory optimisation used below.
%pip install -q "transformers>=4.45" "datasets>=2.19" "accelerate>=0.30" matplotlib
# Only needed if you set cfg.use_lora=True (recommended for >=1.5B models on a 16 GB GPU):
%pip install -q "peft>=0.11"
print("dependencies installed")

In [ ]:
import os, re, gc, random
from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# T4 (Turing) has no bf16 -> we fall back to fp16. A100/L4/H100 use bf16.
DTYPE = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

print("device:", DEVICE, "| dtype:", DTYPE)
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory/1e9:.1f} GB")

def set_seed(s: int):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

In [ ]:
# === Config: every paper's trick is a flag here ==================================
# Read this as the control panel. Each field maps to a concept in the GRPO++ overview;
# the "PAPER" comments tell you which one. Defaults = a sensible DAPO-flavoured setup
# that runs on a free Colab T4 with Qwen2.5-0.5B.

@dataclass
class GRPOConfig:
    # ---- model / data ----
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"  # 0.5B fits free T4; see infra cell for bigger
    use_lora: bool = False                 # set True for >=1.5B on 16 GB (cuts optimiser memory)
    dataset_name: str = "openai/gsm8k"
    dataset_config: str = "main"
    n_train: int = 2000                    # subsample of GSM8K train for speed
    n_eval: int = 200                      # held-out eval (greedy) -> the honest metric

    # ---- sampling / group ----
    prompts_per_step: int = 4              # B : distinct prompts per RL step
    group_size: int = 8                    # G : completions per prompt (GRPO needs G high)
    max_prompt_len: int = 320
    max_completion_len: int = 200          # short on purpose -> low memory + fast on T4
    temperature: float = 1.0
    top_p: float = 1.0

    # ---- optimisation ----
    lr: float = 1e-6                       # RL LRs are tiny
    num_steps: int = 200
    mu: int = 2                            # PPO inner epochs per batch. >1 -> clipping ACTIVE
    grad_clip: float = 0.2
    micro_bs: int = 4                      # completions per forward/backward chunk (memory lever)
    gradient_checkpointing: bool = True

    # ---- DAPO [1]: clip-higher ----
    eps_low: float = 0.2                   # do NOT raise (collapses sampling space)
    eps_high: float = 0.28                 # raise this to fight entropy collapse

    # ---- loss aggregation (DAPO token-level / Dr.GRPO const-norm) ----
    loss_agg: str = "dapo"                 # "grpo" (per-seq mean) | "dapo" (token-level) | "dr_grpo" (const)

    # ---- Dr. GRPO [3]: advantage ----
    scale_advantage_by_std: bool = True    # False = Dr.GRPO (drop /std, removes difficulty bias)

    # ---- objective variant ----
    loss_variant: str = "grpo"             # "grpo" | "gspo" (seq-level ratio) | "cispo" (soft clip)

    # ---- KL ----
    beta_kl: float = 0.0                   # 0.0 = no-KL (standard for reasoning RL)

    # ---- DAPO [1]: dynamic sampling + overlong handling ----
    dynamic_sampling: bool = True          # drop groups where every reward is equal (zero advantage)
    mask_truncated: bool = True            # truncated completions contribute 0 to the loss
    soft_overlong: bool = False            # soft length penalty in the punishment interval
    overlong_cache: int = 40               # L_cache (in tokens) for the soft penalty ramp

    # ---- TIS [4]: sampler != learner correction ----
    use_tis: bool = False                  # only meaningful when rollouts come from a different engine (vLLM)
    tis_cap: float = 2.0                   # truncation cap rho on the importance ratio

    # ---- bookkeeping ----
    eval_every: int = 50
    seed: int = 0

cfg = GRPOConfig()
set_seed(cfg.seed)
print(cfg)

In [ ]:
# === Load policy (and optional reference) =======================================
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
# Left-padding is required: we generate from a batch of prompts and then slice the
# completion tokens off the right edge, so all prompts must end at the same index.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_policy():
    m = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=DTYPE,
        attn_implementation="eager",   # avoids flash/sdpa edge cases on Colab T4
    )
    m.to(DEVICE)
    if cfg.gradient_checkpointing:
        m.config.use_cache = False
        m.gradient_checkpointing_enable()
    return m

policy = load_policy()

# LoRA (optional) -- makes 1.5B trainable on a 16 GB GPU by shrinking the optimiser state.
if cfg.use_lora:
    from peft import LoraConfig, get_peft_model
    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    )
    policy = get_peft_model(policy, lora)
    policy.print_trainable_parameters()

# Reference model is ONLY needed when beta_kl > 0.
# With LoRA we don't load a second model -- we get the reference distribution for free by
# disabling the adapter (policy.disable_adapter()), handled later in the step.
ref_model = None
if cfg.beta_kl > 0 and not cfg.use_lora:
    ref_model = load_policy()
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

trainable = [p for p in policy.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg.lr)
print(f"trainable tensors: {len(trainable)} | KL ref model: {'yes' if ref_model is not None else 'no'}")

In [ ]:
# === Data: GSM8K with a format-enforcing system prompt ==========================
# RLVR needs a verifiable target. GSM8K gives a gold numeric answer after "####".
# We ask the model to put its final answer inside <answer>...</answer> so parsing is robust,
# and to think inside <reasoning>...</reasoning> (the long-CoT the model learns to use).

SYSTEM_PROMPT = (
    "You are a careful math problem solver. Think step by step, then give the final "
    "numeric answer.\n"
    "Respond in exactly this format:\n"
    "<reasoning>\nyour step-by-step working\n</reasoning>\n"
    "<answer>\nfinal number only\n</answer>"
)

def extract_gold(answer_field: str) -> str:
    # GSM8K gold lives after the '####' marker.
    after = answer_field.split("####")[-1]
    return after.strip().replace(",", "")

def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    # add_generation_prompt=True appends the assistant header so the model continues as assistant.
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

raw = load_dataset(cfg.dataset_name, cfg.dataset_config)
train_raw = raw["train"].select(range(min(cfg.n_train, len(raw["train"]))))
eval_raw  = raw["test"].select(range(min(cfg.n_eval, len(raw["test"]))))

train_data = [{"prompt": build_prompt(r["question"]), "gold": extract_gold(r["answer"])}
              for r in train_raw]
eval_data  = [{"prompt": build_prompt(r["question"]), "gold": extract_gold(r["answer"])}
              for r in eval_raw]

print(f"train: {len(train_data)} | eval: {len(eval_data)}")
print("\n--- example prompt ---\n", train_data[0]["prompt"][:600], "\n...\ngold:", train_data[0]["gold"])

In [ ]:
# === Verifiable rewards =========================================================
# RLVR reward = correctness (the verifiable signal) + a small format bonus that teaches
# the model to emit a parseable answer early in training. This mirrors how real recipes
# combine a hard correctness check with light format shaping.

_ANS_TAG   = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.DOTALL)
_BOXED     = re.compile(r"\\boxed\{([^}]*)\}")
_LAST_NUM  = re.compile(r"-?\d+(?:\.\d+)?")

def _to_float(s: str):
    s = s.strip().replace(",", "").replace("$", "").rstrip(".")
    try:
        return float(s)
    except ValueError:
        return None

def extract_pred(text: str):
    # Preference order: <answer> tag -> \boxed{} -> last number in the text.
    m = _ANS_TAG.search(text)
    if m:
        v = _to_float(m.group(1))
        if v is not None:
            return v
    m = _BOXED.search(text)
    if m:
        v = _to_float(m.group(1))
        if v is not None:
            return v
    nums = _LAST_NUM.findall(text)
    return _to_float(nums[-1]) if nums else None

def correctness_reward(text: str, gold: str) -> float:
    pred = extract_pred(text)
    gold_f = _to_float(gold)
    if pred is None or gold_f is None:
        return 0.0
    return 1.0 if abs(pred - gold_f) < 1e-4 else 0.0

def format_reward(text: str) -> float:
    # Small shaping bonus: reward presence of both tags. Kept small so correctness dominates.
    bonus = 0.0
    if "<reasoning>" in text and "</reasoning>" in text:
        bonus += 0.1
    if "<answer>" in text and "</answer>" in text:
        bonus += 0.1
    return bonus

def reward_fn(text: str, gold: str) -> float:
    # A correct, well-formatted answer scores ~1.2; correctness alone is the load-bearing part.
    return correctness_reward(text, gold) + format_reward(text)

def soft_overlong_penalty(gen_len: int, cfg) -> float:
    # DAPO [1] soft length penalty over the interval [L_max - L_cache, L_max].
    # 0 below the interval, ramps linearly to -1 at L_max. Added on top of reward_fn.
    L_max = cfg.max_completion_len
    L_cache = cfg.overlong_cache
    start = L_max - L_cache
    if gen_len <= start:
        return 0.0
    if gen_len >= L_max:
        return -1.0
    return -(gen_len - start) / float(L_cache)

# quick sanity check
_demo = "<reasoning>2+2=4</reasoning><answer>4</answer>"
print("reward on correct+formatted:", reward_fn(_demo, "4"))
print("reward on wrong:", reward_fn("the answer is 5", "4"))

In [ ]:
# === Core mechanics: log-probs, generation, advantages ==========================
# These are the pieces the GRPO loss is built from. The fiddly bits (memory, masking,
# version robustness) are commented because they're exactly where real implementations bite.

def _forward_logits(model, input_ids, attention_mask, num_keep):
    # Only materialise logits for the last `num_keep` positions. Full-vocab logits over
    # B*G long sequences would OOM a T4. The kwarg name changed across transformers versions,
    # so we try the new one, then the old one, then fall back to full logits.
    try:
        out = model(input_ids=input_ids, attention_mask=attention_mask, logits_to_keep=num_keep + 1)
    except TypeError:
        try:
            out = model(input_ids=input_ids, attention_mask=attention_mask, num_logits_to_keep=num_keep + 1)
        except TypeError:
            out = model(input_ids=input_ids, attention_mask=attention_mask)
    return out.logits

def get_per_token_logps(model, input_ids, attention_mask, comp_len):
    # Log-prob of each *completion* token under `model`.
    # Logits at position t predict token t+1, so we shift: logits[:, :-1] line up with input_ids[:, 1:].
    logits = _forward_logits(model, input_ids, attention_mask, comp_len)
    logits = logits[:, :-1, :]
    target = input_ids[:, 1:]
    # keep only the completion tail
    logits = logits[:, -comp_len:, :]
    target = target[:, -comp_len:]
    logps = torch.log_softmax(logits.float(), dim=-1)          # fp32 for numerical stability
    return torch.gather(logps, dim=-1, index=target.unsqueeze(-1)).squeeze(-1)  # (B, comp_len)

def entropy_from_logits(model, input_ids, attention_mask, comp_len):
    # Token-level entropy of the next-token distribution over the completion span.
    # Health metric: we want this to stay in a healthy band, not collapse.
    logits = _forward_logits(model, input_ids, attention_mask, comp_len)
    logits = logits[:, :-1, :][:, -comp_len:, :].float()
    logp = torch.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -(p * logp).sum(-1)                                  # (B, comp_len)

def masked_mean(x, mask, dim=None):
    mask = mask.to(x.dtype)
    if dim is None:
        return (x * mask).sum() / mask.sum().clamp(min=1.0)
    return (x * mask).sum(dim) / mask.sum(dim).clamp(min=1.0)

@torch.no_grad()
def generate_groups(prompts, cfg):
    # Sample G completions per prompt. Returns padded full sequences + a completion mask.
    # Generation wants the KV cache ON and checkpointing OFF (speed); training wants the
    # opposite (memory). We flip the flags here and flip them back in the training step.
    policy.config.use_cache = True
    if cfg.gradient_checkpointing:
        policy.gradient_checkpointing_disable()

    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                    max_length=cfg.max_prompt_len)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    prompt_len = enc["input_ids"].shape[1]

    gen = policy.generate(
        **enc,
        max_new_tokens=cfg.max_completion_len,
        do_sample=True,
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        num_return_sequences=cfg.group_size,
        pad_token_id=tokenizer.pad_token_id,
    )
    # gen: (B*G, prompt_len + comp_len)
    comp_ids = gen[:, prompt_len:]
    comp_len = comp_ids.shape[1]

    # Build a completion mask that is 1 up to and including the first EOS, 0 after.
    is_eos = (comp_ids == tokenizer.eos_token_id)
    has_eos = is_eos.any(dim=1)
    first_eos = is_eos.float().argmax(dim=1)                    # 0 if no EOS -> fixed next line
    idx = torch.arange(comp_len, device=DEVICE).unsqueeze(0)
    # where there is an EOS, keep tokens at positions <= first_eos; else keep all positions
    keep_upto = torch.where(has_eos, first_eos, torch.full_like(first_eos, comp_len - 1))
    comp_mask = (idx <= keep_upto.unsqueeze(1)).long()          # (B*G, comp_len)

    truncated = ~has_eos                                         # never emitted EOS -> hit the cap
    prompt_mask = enc["attention_mask"]                          # (B, prompt_len)
    full_attn = torch.cat(
        [prompt_mask.repeat_interleave(cfg.group_size, dim=0), comp_mask], dim=1
    )
    return {
        "full_ids": gen, "full_attn": full_attn,
        "comp_mask": comp_mask, "comp_len": comp_len,
        "truncated": truncated, "prompt_len": prompt_len,
    }

def compute_group_advantages(rewards, cfg):
    # rewards: (n_prompts, G). Center within each group; optionally divide by std (Dr.GRPO drops this).
    mean = rewards.mean(dim=1, keepdim=True)
    adv = rewards - mean
    if cfg.scale_advantage_by_std:
        std = rewards.std(dim=1, keepdim=True)
        adv = adv / (std + 1e-6)
    return adv                                                  # (n_prompts, G)

In [ ]:
# === The loss: GRPO / GSPO / CISPO in one place =================================
# This is the heart of the demo. Given new vs old log-probs (and optionally reference /
# sampler log-probs), it returns *summed* pieces so the caller can normalise correctly
# under micro-batching. The three variants differ only in how the ratio is used.
#
# Returns per call (all summed over the chunk):
#   per_seq_loss : (chunk,)  loss summed over tokens, per sequence
#   per_seq_tok  : (chunk,)  number of live tokens per sequence (for per-seq mean variants)
#   clip_count   : scalar    how many tokens/seqs were clipped (health metric)
#   n_tok        : scalar    total live tokens in the chunk
#   kl_sum       : scalar    summed KL (0 when beta_kl == 0)

def grpo_chunk_loss(new_logps, old_logps, sampler_logps, ref_logps, adv, mask, cfg):
    # new/old/sampler/ref_logps, mask: (chunk, comp_len); adv: (chunk,) one advantage per sequence
    adv = adv.unsqueeze(1)                                   # (chunk, 1) broadcast over tokens
    log_ratio = new_logps - old_logps                       # log( pi_new / pi_old )

    if cfg.loss_variant == "gspo":
        # GSPO [6]: ONE importance ratio per sequence (length-normalised), then clip it.
        seq_log_ratio = masked_mean(log_ratio, mask, dim=1)         # (chunk,)
        s = torch.exp(seq_log_ratio)
        s_clip = torch.clamp(s, 1 - cfg.eps_low, 1 + cfg.eps_high)
        a = adv.squeeze(1)
        unclipped = s * a
        clipped = s_clip * a
        per_seq_obj = torch.minimum(unclipped, clipped)            # pessimistic bound
        per_seq_loss = -per_seq_obj                                # (chunk,) one value per seq
        per_seq_tok = torch.ones_like(per_seq_loss)
        clip_count = (clipped < unclipped).sum()
        # GSPO clips whole SEQUENCES, so report clip_frac per-sequence (not per-token).
        n_tok = torch.tensor(float(per_seq_loss.shape[0]), device=new_logps.device)

    elif cfg.loss_variant == "cispo":
        # CISPO [8]: REINFORCE-style. The importance weight is CLIPPED and DETACHED (stop-grad),
        # so even "clipped" fork tokens still pass gradient through new_logps -> they keep
        # contributing across all inner updates. This is the key difference from PPO clipping.
        ratio = torch.exp(log_ratio)
        w = torch.clamp(ratio, 1 - cfg.eps_low, 1 + cfg.eps_high).detach()
        token_loss = -(w * adv * new_logps)                        # (chunk, comp_len)
        token_loss = token_loss * mask
        per_seq_loss = token_loss.sum(dim=1)
        per_seq_tok = mask.sum(dim=1)
        clip_count = ((ratio < 1 - cfg.eps_low) | (ratio > 1 + cfg.eps_high)).float().mul(mask).sum()
        n_tok = mask.sum()

    else:  # "grpo" -> standard PPO-style clipped surrogate, token level
        ratio = torch.exp(log_ratio)
        unclipped = ratio * adv
        clipped = torch.clamp(ratio, 1 - cfg.eps_low, 1 + cfg.eps_high) * adv
        token_obj = torch.minimum(unclipped, clipped)              # (chunk, comp_len)
        token_loss = -(token_obj * mask)
        per_seq_loss = token_loss.sum(dim=1)
        per_seq_tok = mask.sum(dim=1)
        clip_count = (clipped < unclipped).float().mul(mask).sum()
        n_tok = mask.sum()

    # ---- optional KL penalty (k3 estimator), added token-level ----
    kl_sum = torch.tensor(0.0, device=new_logps.device)
    if cfg.beta_kl > 0 and ref_logps is not None:
        kl_tok = torch.exp(ref_logps - new_logps) - (ref_logps - new_logps) - 1.0
        kl_tok = kl_tok * mask
        per_seq_loss = per_seq_loss + cfg.beta_kl * kl_tok.sum(dim=1)
        kl_sum = kl_tok.sum()

    # ---- optional TIS [4] correction (sampler != learner) ----
    # No-op in this single-engine notebook (sampler_logps == old_logps -> weight 1).
    # On a vLLM rollout setup, sampler_logps would be the vLLM log-probs and this term
    # corrects the off-policy gap. Weight is truncated and detached.
    if cfg.use_tis and sampler_logps is not None:
        tis_w = torch.clamp(torch.exp(old_logps - sampler_logps), max=cfg.tis_cap).detach()
        tis_seq = masked_mean(tis_w, mask, dim=1)                  # (chunk,)
        per_seq_loss = per_seq_loss * tis_seq

    return per_seq_loss, per_seq_tok, clip_count, n_tok, kl_sum

In [ ]:
# === One RL step ================================================================
# Sequence per step:
#   1. sample groups (oversample if dynamic sampling is on)
#   2. score rewards -> reshape to (n_prompts, G)
#   3. dynamic sampling: drop groups whose rewards are all-equal (zero advantage)
#   4. compute advantages on the KEPT groups
#   5. snapshot old (behaviour) log-probs once
#   6. mu inner epochs: micro-batch the completions, compute the loss, normalise, step
#
# The normalisation is the subtle part. We accumulate a *summed* numerator across micro
# chunks and divide by a single global denominator so token-level / constant-length
# normalisation stays exactly correct under chunking.

def _denominator(loss_agg, loss_variant, n_seqs, total_tokens, cfg):
    # What we divide the summed objective by -> this is where DAPO vs Dr.GRPO vs per-seq differ.
    if loss_variant in ("gspo", "cispo"):
        # gspo already produced one term per sequence; cispo is token-level summed.
        return float(n_seqs) if loss_variant == "gspo" else float(max(total_tokens, 1))
    if loss_agg == "dapo":          # token-level: divide by total live tokens in the batch
        return float(max(total_tokens, 1))
    if loss_agg == "dr_grpo":       # Dr.GRPO [3]: divide by a CONSTANT (n_seqs * max_len)
        return float(n_seqs * cfg.max_completion_len)
    return float(max(n_seqs, 1))    # "grpo": per-sequence mean (handled with per_seq_tok below)

def rl_step(step_idx, history):
    policy.train()
    # ---- 1. choose prompts (oversample for dynamic sampling) ----
    over = 2 if cfg.dynamic_sampling else 1
    n_sample = cfg.prompts_per_step * over
    batch = random.sample(train_data, min(n_sample, len(train_data)))
    prompts = [b["prompt"] for b in batch]
    golds = [b["gold"] for b in batch]

    g = generate_groups(prompts, cfg)
    G = cfg.group_size
    comp_len = g["comp_len"]
    n_prompts = len(prompts)

    # ---- 2. rewards (decode -> verify) ----
    texts = tokenizer.batch_decode(
        g["full_ids"][:, g["prompt_len"]:], skip_special_tokens=True
    )
    gold_per_row = [golds[i // G] for i in range(n_prompts * G)]
    rewards = torch.tensor(
        [reward_fn(t, gold) for t, gold in zip(texts, gold_per_row)],
        dtype=torch.float32, device=DEVICE,
    )
    if cfg.soft_overlong:
        gen_lens = g["comp_mask"].sum(dim=1)
        pen = torch.tensor(
            [soft_overlong_penalty(int(l), cfg) for l in gen_lens],
            dtype=torch.float32, device=DEVICE,
        )
        rewards = rewards + pen

    R_full = rewards.view(n_prompts, G)                       # (n_prompts, G)

    # ---- 3. dynamic sampling: keep only groups with reward spread ----
    if cfg.dynamic_sampling:
        spread = R_full.std(dim=1)
        order = torch.argsort(spread, descending=True)
        keep = order[spread[order] > 1e-6][: cfg.prompts_per_step]
        if keep.numel() == 0:                                 # fallback: nothing had spread
            keep = order[: cfg.prompts_per_step]
    else:
        keep = torch.arange(min(cfg.prompts_per_step, n_prompts), device=DEVICE)

    keep = keep.tolist()
    # ALIGNMENT: rows for kept groups, in the SAME order we index R_keep below.
    row_idx = [grp * G + k for grp in keep for k in range(G)]
    row_idx_t = torch.tensor(row_idx, device=DEVICE)
    R_keep = R_full[keep]                                     # (n_keep, G) -- same order as row_idx

    full_ids = g["full_ids"][row_idx_t]
    full_attn = g["full_attn"][row_idx_t]
    comp_mask = g["comp_mask"][row_idx_t].clone()
    truncated = g["truncated"][row_idx_t]

    # ---- DAPO overlong masking: truncated completions contribute zero ----
    if cfg.mask_truncated:
        comp_mask[truncated] = 0

    # ---- 4. advantages on kept groups, flattened to per-row ----
    adv_grp = compute_group_advantages(R_keep, cfg)           # (n_keep, G)
    adv = adv_grp.reshape(-1)                                 # (n_keep*G,) aligned with row_idx
    n_seqs = adv.shape[0]

    # ---- 5. snapshot old log-probs + reference + sampler (once) ----
    # We micro-batch these forwards: full-vocab log_softmax over the whole batch at once
    # would allocate several GB (vocab ~151k) and can OOM a 16 GB T4. Chunking keeps peak
    # memory at micro_bs sequences regardless of group_size.
    with torch.no_grad():
        if cfg.gradient_checkpointing:
            policy.gradient_checkpointing_enable()
        policy.config.use_cache = False

        old_chunks, ent_chunks, ref_chunks = [], [], []
        for s in range(0, n_seqs, cfg.micro_bs):
            e = min(s + cfg.micro_bs, n_seqs)
            old_chunks.append(get_per_token_logps(policy, full_ids[s:e], full_attn[s:e], comp_len))
            ent_chunks.append(entropy_from_logits(policy, full_ids[s:e], full_attn[s:e], comp_len))
            if cfg.beta_kl > 0:
                if cfg.use_lora:
                    with policy.disable_adapter():
                        ref_chunks.append(get_per_token_logps(policy, full_ids[s:e], full_attn[s:e], comp_len))
                elif ref_model is not None:
                    ref_chunks.append(get_per_token_logps(ref_model, full_ids[s:e], full_attn[s:e], comp_len))

        old_logps = torch.cat(old_chunks, dim=0)
        ent = torch.cat(ent_chunks, dim=0)
        sampler_logps = old_logps                             # single engine -> sampler == old
        ref_logps = torch.cat(ref_chunks, dim=0) if ref_chunks else None

    total_tokens = int(comp_mask.sum().item())
    denom = _denominator(cfg.loss_agg, cfg.loss_variant, n_seqs, total_tokens, cfg)

    # ---- 6. mu inner epochs, micro-batched over completions ----
    clip_sum = 0.0; tok_sum = 0.0; kl_sum = 0.0; loss_val = 0.0; grad_norm = 0.0
    for _ in range(cfg.mu):
        optimizer.zero_grad()
        epoch_clip = 0.0; epoch_tok = 0.0; epoch_kl = 0.0; epoch_loss = 0.0
        for s in range(0, n_seqs, cfg.micro_bs):
            e = min(s + cfg.micro_bs, n_seqs)
            new_logps = get_per_token_logps(policy, full_ids[s:e], full_attn[s:e], comp_len)
            ps_loss, ps_tok, c_clip, c_tok, c_kl = grpo_chunk_loss(
                new_logps,
                old_logps[s:e],
                sampler_logps[s:e],
                None if ref_logps is None else ref_logps[s:e],
                adv[s:e],
                comp_mask[s:e],
                cfg,
            )
            # chunk numerator -> matches the global denom chosen above
            if cfg.loss_variant in ("gspo", "cispo"):
                chunk_num = ps_loss.sum()
            elif cfg.loss_agg == "grpo":
                chunk_num = (ps_loss / ps_tok.clamp(min=1)).sum()   # per-seq mean, summed
            else:                                                   # dapo / dr_grpo
                chunk_num = ps_loss.sum()
            (chunk_num / denom).backward()                          # accumulate grads across chunks

            epoch_loss += float(chunk_num.item())
            epoch_clip += float(c_clip.item())
            epoch_tok  += float(c_tok.item())
            epoch_kl   += float(c_kl.item())

        grad_norm = float(torch.nn.utils.clip_grad_norm_(trainable, cfg.grad_clip))
        optimizer.step()
        loss_val = epoch_loss / denom
        clip_sum = epoch_clip; tok_sum = epoch_tok; kl_sum = epoch_kl

    # ---- metrics (report on FULL sampled R for an unbiased health signal) ----
    full_mask_lens = g["comp_mask"].sum(dim=1).float()
    rec = {
        "step": step_idx,
        "reward": float(R_full.mean().item()),                # unbiased: all sampled groups
        "reward_kept": float(R_keep.mean().item()),           # what we actually trained on
        "comp_len": float(full_mask_lens.mean().item()),
        "entropy": float(masked_mean(ent, comp_mask).item()),
        "clip_frac": float(clip_sum / max(tok_sum, 1.0)),
        "zero_adv_frac": float((R_full.std(dim=1) <= 1e-6).float().mean().item()),
        "kl": float(kl_sum / max(total_tokens, 1)),
        "grad_norm": grad_norm,
        "loss": loss_val,
        "n_groups": len(keep),
    }
    history.append(rec)
    return rec

In [ ]:
# === Held-out evaluation (the honest metric) ====================================
# Training reward is biased (dynamic sampling, format bonus, overfitting). The number that
# actually matters is greedy accuracy on held-out prompts. We decode with do_sample=False.

@torch.no_grad()
def evaluate(n=None, greedy=True, bs=16):
    policy.eval()
    policy.config.use_cache = True
    if cfg.gradient_checkpointing:
        policy.gradient_checkpointing_disable()

    data = eval_data if n is None else eval_data[:n]
    correct = 0
    for s in range(0, len(data), bs):
        chunk = data[s:s + bs]
        prompts = [c["prompt"] for c in chunk]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                        max_length=cfg.max_prompt_len)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        plen = enc["input_ids"].shape[1]
        gen = policy.generate(
            **enc,
            max_new_tokens=cfg.max_completion_len,
            do_sample=not greedy,                 # greedy -> no temperature/top_p needed
            pad_token_id=tokenizer.pad_token_id,
        )
        texts = tokenizer.batch_decode(gen[:, plen:], skip_special_tokens=True)
        for t, c in zip(texts, chunk):
            if correctness_reward(t, c["gold"]) >= 1.0:
                correct += 1
    acc = correct / max(len(data), 1)
    policy.train()
    return acc

print("evaluate() ready")

In [ ]:
# === Train ======================================================================
# Baseline eval, then the RL loop. We print a sample completion at each eval so you can
# watch the format and reasoning quality change. Expect reward + format to move within
# ~50-100 steps on the 0.5B model; absolute GSM8K accuracy gains at 0.5B are modest
# (mostly format/parse wins -- the bigger the model, the bigger the reasoning gain).

from tqdm.auto import tqdm

history = []

base_acc = evaluate(n=min(64, cfg.n_eval))
print(f"[baseline] greedy GSM8K acc on {min(64, cfg.n_eval)} held-out: {base_acc:.3f}\n")

pbar = tqdm(range(1, cfg.num_steps + 1))
for step in pbar:
    rec = rl_step(step, history)
    pbar.set_postfix({
        "reward": f"{rec['reward']:.2f}",
        "len": f"{rec['comp_len']:.0f}",
        "ent": f"{rec['entropy']:.2f}",
        "clip": f"{rec['clip_frac']:.2f}",
        "0adv": f"{rec['zero_adv_frac']:.2f}",
    })

    if step % cfg.eval_every == 0 or step == cfg.num_steps:
        acc = evaluate(n=min(64, cfg.n_eval))
        # show one sample so you can eyeball reasoning/format quality
        sample = random.choice(train_data)
        gg = generate_groups([sample["prompt"]], cfg)
        out = tokenizer.batch_decode(gg["full_ids"][:1, gg["prompt_len"]:],
                                     skip_special_tokens=True)[0]
        print(f"\n[step {step}] held-out acc: {acc:.3f} | gold: {sample['gold']}")
        print("sample completion:\n", out[:400])
        rec["eval_acc"] = acc

    if DEVICE == "cuda":
        gc.collect()
        torch.cuda.empty_cache()

final_acc = evaluate(n=min(128, cfg.n_eval))
print(f"\n[final] greedy GSM8K acc: {final_acc:.3f} (baseline was {base_acc:.3f})")

In [ ]:
# === Health dashboard ===========================================================
# The six curves the overview tells you to watch. Read them together:
#  - reward up & smooth         -> learning is stable
#  - completion length          -> should rise as long-CoT develops (not explode)
#  - entropy in a healthy band  -> exploration alive, not collapsed
#  - clip fraction              -> how often the trust region binds
#  - zero-advantage fraction    -> how much of each batch dynamic sampling would waste
#  - grad norm                  -> spikes = instability
import matplotlib.pyplot as plt

def plot_history(history):
    if not history:
        print("no history yet -- run training first"); return
    steps = [h["step"] for h in history]
    panels = [
        ("reward",        "training reward (full sampled groups)"),
        ("comp_len",      "mean completion length (tokens)"),
        ("entropy",       "policy entropy"),
        ("clip_frac",     "clip fraction"),
        ("zero_adv_frac", "zero-advantage group fraction"),
        ("grad_norm",     "grad norm"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, (key, title) in zip(axes.flat, panels):
        ax.plot(steps, [h[key] for h in history])
        ax.set_title(title); ax.set_xlabel("step"); ax.grid(alpha=0.3)
    # overlay held-out accuracy points on the reward panel where available
    ev = [(h["step"], h["eval_acc"]) for h in history if "eval_acc" in h]
    if ev:
        ax2 = axes.flat[0].twinx()
        ax2.scatter([e[0] for e in ev], [e[1] for e in ev], color="crimson", marker="o", label="eval acc")
        ax2.set_ylabel("held-out acc", color="crimson")
    plt.tight_layout(); plt.show()

plot_history(history)

## Ablations: turn each paper's trick on and off

This is where the notebook earns the word *advanced*. Every modification from the overview
is a config flag, so you can run controlled A/B comparisons and watch the health curves
respond — exactly the iterate-and-monitor workflow the article describes.

Each preset below changes **one idea** relative to `vanilla_grpo` (except the combined
recipes at the end). Run a couple, then `overlay()` their curves.

| preset | what changes | isolates |
|--------|--------------|----------|
| `vanilla_grpo` | per-seq loss, std-norm advantage, `eps_high=0.2`, `mu=1`, no dynamic/mask | the baseline with the known issues |
| `clip_higher` | `eps_high=0.28` | DAPO entropy-collapse fix [1] |
| `token_level` | `loss_agg="dapo"` | DAPO length-bias fix [1] |
| `dr_grpo` | `scale_advantage_by_std=False` + constant-length norm | Dr.GRPO bias fixes [3] |
| `dynamic_sample` | `dynamic_sampling=True` | DAPO sample-efficiency fix [1] |
| `overlong_mask` | `mask_truncated=True` | DAPO truncation handling [1] |
| `gspo` | `loss_variant="gspo"` | sequence-level ratio [6] |
| `cispo` | `loss_variant="cispo"`, `mu=4` | soft clipping keeps fork tokens [8] |
| `dapo_full` | clip-higher + token-level + dynamic + mask | the full DAPO system [1] |
| `olmo3_style` | Dr.GRPO advantage + token-level + dynamic + clip-higher + TIS | the combined recipe [5] |

> **Cost note:** each preset re-trains a fresh model, so on a free T4 run them with a small
> `num_steps` (e.g. 40–60) and small `n_train`. The *shape* of the curves is what teaches
> the lesson, not the final accuracy. `cispo` uses `mu=4` because its whole point only shows
> up across many inner updates (where PPO clipping would have frozen the fork tokens).

In [ ]:
# === Ablation harness ===========================================================
# Each preset is a dict of overrides on top of the defaults. make_cfg() builds a fresh
# config; to actually run a preset you re-instantiate the model and call the training loop
# with that cfg. We keep this explicit (rather than hidden in a function) so the mechanics
# stay visible. Below we give a ready-to-run runner you can call per preset.
import copy

PRESETS = {
    "vanilla_grpo": dict(eps_high=0.2, loss_agg="grpo", scale_advantage_by_std=True,
                         mu=1, dynamic_sampling=False, mask_truncated=False,
                         loss_variant="grpo", use_tis=False),
    "clip_higher":  dict(eps_high=0.28, loss_agg="grpo", scale_advantage_by_std=True,
                         mu=2, dynamic_sampling=False, mask_truncated=False, loss_variant="grpo"),
    "token_level":  dict(eps_high=0.2, loss_agg="dapo", scale_advantage_by_std=True,
                         mu=2, dynamic_sampling=False, mask_truncated=False, loss_variant="grpo"),
    "dr_grpo":      dict(eps_high=0.2, loss_agg="dr_grpo", scale_advantage_by_std=False,
                         mu=2, dynamic_sampling=False, mask_truncated=False, loss_variant="grpo"),
    "dynamic_sample": dict(eps_high=0.2, loss_agg="grpo", scale_advantage_by_std=True,
                           mu=2, dynamic_sampling=True, mask_truncated=False, loss_variant="grpo"),
    "overlong_mask": dict(eps_high=0.2, loss_agg="grpo", scale_advantage_by_std=True,
                          mu=2, dynamic_sampling=False, mask_truncated=True,
                          soft_overlong=True, loss_variant="grpo"),
    "gspo":         dict(eps_high=0.28, loss_agg="dapo", scale_advantage_by_std=True,
                         mu=2, dynamic_sampling=True, mask_truncated=True, loss_variant="gspo"),
    "cispo":        dict(eps_high=0.28, loss_agg="dapo", scale_advantage_by_std=True,
                         mu=4, dynamic_sampling=True, mask_truncated=True, loss_variant="cispo"),
    "dapo_full":    dict(eps_high=0.28, loss_agg="dapo", scale_advantage_by_std=True,
                         mu=2, dynamic_sampling=True, mask_truncated=True,
                         soft_overlong=True, loss_variant="grpo"),
    "olmo3_style":  dict(eps_high=0.28, loss_agg="dapo", scale_advantage_by_std=False,
                         mu=2, dynamic_sampling=True, mask_truncated=True,
                         use_tis=True, loss_variant="grpo"),
}

def make_cfg(name, **extra):
    base = GRPOConfig()
    for k, v in PRESETS[name].items():
        setattr(base, k, v)
    for k, v in extra.items():
        setattr(base, k, v)
    return base

# --- runner: trains a FRESH model under a given preset and returns its history ---
# NOTE: this rebinds the globals `cfg`, `policy`, `optimizer`, etc. so it reuses the exact
# same step/eval code defined above. Call with small num_steps on a free GPU.
def run_preset(name, num_steps=50, n_train=400, **extra):
    global cfg, policy, ref_model, optimizer, trainable
    cfg = make_cfg(name, num_steps=num_steps, n_train=n_train, **extra)
    set_seed(cfg.seed)

    # rebuild data subset for the (possibly) smaller n_train
    td = [{"prompt": build_prompt(r["question"]), "gold": extract_gold(r["answer"])}
          for r in train_raw.select(range(min(cfg.n_train, len(train_raw))))]
    train_data.clear(); train_data.extend(td)

    # fresh model + optimiser
    policy = load_policy()
    if cfg.use_lora:
        from peft import LoraConfig, get_peft_model
        lc = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                        task_type="CAUSAL_LM",
                        target_modules=["q_proj","k_proj","v_proj","o_proj",
                                        "gate_proj","up_proj","down_proj"])
        policy = get_peft_model(policy, lc)
    ref_model = None
    if cfg.beta_kl > 0 and not cfg.use_lora:
        ref_model = load_policy(); ref_model.eval()
        for p in ref_model.parameters(): p.requires_grad_(False)
    trainable = [p for p in policy.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=cfg.lr)

    hist = []
    for step in range(1, cfg.num_steps + 1):
        rl_step(step, hist)
        if DEVICE == "cuda":
            gc.collect(); torch.cuda.empty_cache()
    return hist

def overlay(histories: dict, key="reward"):
    # histories: {preset_name: history_list}
    plt.figure(figsize=(9, 5))
    for name, h in histories.items():
        plt.plot([r["step"] for r in h], [r[key] for r in h], label=name)
    plt.title(f"ablation overlay: {key}"); plt.xlabel("step"); plt.ylabel(key)
    plt.legend(); plt.grid(alpha=0.3); plt.show()

# --- example (uncomment to run; each call retrains a fresh model) ---
# results = {}
# for name in ["vanilla_grpo", "clip_higher", "dapo_full"]:
#     print(f"=== running {name} ===")
#     results[name] = run_preset(name, num_steps=50, n_train=400)
# overlay(results, key="reward")
# overlay(results, key="entropy")
print("ablation presets ready:", list(PRESETS))

## Map: paper → this notebook's flag → TRL's `GRPOConfig` knob

Once you understand the mechanics from the code above, here is how each idea translates to
the production library (`trl`). The from-scratch loop and TRL line up almost 1:1.

| Concept | Paper | This notebook | `trl.GRPOConfig` |
|---|---|---|---|
| Group-relative advantage, no critic | GRPO [13] | `compute_group_advantages` | built in (`num_generations=G`) |
| Clip-higher (decoupled ε) | DAPO [1] | `eps_low`, `eps_high` | `epsilon`, `epsilon_high` |
| Token-level loss | DAPO [1] | `loss_agg="dapo"` | `loss_type="dapo"` |
| Constant-length norm | Dr.GRPO [3] | `loss_agg="dr_grpo"` | `loss_type="dr_grpo"` |
| Drop std-normalisation | Dr.GRPO [3] | `scale_advantage_by_std=False` | `scale_rewards=False` |
| Dynamic sampling | DAPO [1] | `dynamic_sampling=True` | *manual* (not a built-in flag) |
| Overlong masking | DAPO [1] | `mask_truncated=True` | `mask_truncated_completions=True` |
| Soft overlong penalty | DAPO [1] | `soft_overlong=True` | `get_soft_overlong_punishment` reward |
| No-KL training | common | `beta_kl=0.0` | `beta=0.0` |
| Sequence-level ratio | GSPO [6] | `loss_variant="gspo"` | `importance_sampling_level="sequence"` |
| Soft token clipping | CISPO [8] | `loss_variant="cispo"` | *no direct flag* (SAPO is the closest) |
| TIS (sampler≠learner) | TIS [4] | `use_tis=True`, `tis_cap` | `vllm_importance_sampling_correction=True`, `vllm_importance_sampling_cap` |
| PPO inner epochs (μ) | PPO [11] | `mu` | `num_iterations` |

The one thing to internalise: **dynamic sampling is not a TRL toggle** — in production you
implement the oversample-and-filter loop yourself (or use a framework like `verl`/`Oat` that
bakes it in). Everything else is a config line.

## Production path: the same recipe in `trl`

For real training runs you wouldn't hand-roll the loop — you'd use `trl`'s `GRPOTrainer`,
which gives you vLLM-accelerated rollouts, multi-GPU, and the tuned defaults for free. The
code below is a **reference** (not executed in this notebook). It reproduces the DAPO +
Dr.GRPO + GSPO + TIS configuration we built from scratch above.

```python
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset

def correctness_reward(completions, gold, **kwargs):
    # same verifiable signal as the from-scratch reward_fn
    out = []
    for c, g in zip(completions, gold):
        pred = extract_pred(c)              # reuse your parser
        out.append(1.0 if (pred is not None and abs(pred - float(g)) < 1e-4) else 0.0)
    return out

config = GRPOConfig(
    output_dir="grpo-qwen-gsm8k",
    # --- group / batch (article: GRPO needs a high G and a big batch) ---
    num_generations=8,                      # G
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    # --- DAPO [1] ---
    epsilon=0.2,                            # eps_low
    epsilon_high=0.28,                      # clip-higher
    loss_type="dapo",                       # token-level loss
    mask_truncated_completions=True,        # overlong masking
    # --- Dr.GRPO [3] ---
    scale_rewards=False,                    # drop std-normalisation of the advantage
    # (for pure Dr.GRPO loss normalisation use loss_type="dr_grpo")
    # --- GSPO [6] : sequence-level ratio (great for MoE like Qwen3) ---
    importance_sampling_level="sequence",   # use "token" for vanilla GRPO behaviour
    # --- no-KL (standard for reasoning) ---
    beta=0.0,
    # --- PPO inner epochs (article: CISPO-style needs many; DAPO ~2-4) ---
    num_iterations=2,
    # --- TIS [4] : on by default for vLLM rollouts; corrects sampler!=learner ---
    use_vllm=True,
    vllm_importance_sampling_correction=True,
    vllm_importance_sampling_cap=3.0,
    # --- generation budget ---
    max_completion_length=1024,
    temperature=1.0,
    learning_rate=1e-6,
    logging_steps=1,
)

ds = load_dataset("openai/gsm8k", "main", split="train")
ds = ds.map(lambda r: {"prompt": r["question"], "gold": r["answer"].split("####")[-1].strip()})

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    reward_funcs=[correctness_reward],
    args=config,
    train_dataset=ds,
)
trainer.train()
```

**Dynamic sampling** still isn't a flag here — if you want DAPO's oversample-and-filter you
either subclass the trainer or move to `verl`/`Oat`. **CISPO** likewise has no direct TRL
switch; `SAPO`-style soft gating is the nearest in-library idea.

## RunPod or Colab? — sizing the hardware to the run

**Short answer for this notebook's defaults: free Google Colab (T4, 16 GB) is enough.**
Qwen2.5-0.5B-Instruct with short completions (200 tokens), `group_size=8`, and a few
hundred steps fits comfortably and you'll see the curves move. You only need to leave Colab
when you scale the *model*, the *rollout speed*, or the *run length*.

### What runs where

| Setup | Fits on | Notes |
|---|---|---|
| **0.5B, short completions, this demo** | **Free Colab T4 (16 GB)** | the default config; ~smoke test in ~10 min, real run in 1–2 h |
| 1.5B full fine-tune, short ctx | Colab Pro **L4 (24 GB)** / **A100 (40 GB)** | or T4 **with `use_lora=True`** (LoRA shrinks the optimiser state) |
| 1.5B, longer completions (1–2k tok) | A100 40 GB | rollouts get long; this is where speed starts to hurt |
| 7B+ full fine-tune | **RunPod** A100 80 GB / H100, often multi-GPU | not a free-Colab job |
| Fast rollouts via vLLM/SGLang | **RunPod** (A40 / A6000 48 GB, 4090, A100) | dedicated box so you control the inference engine |

### When RunPod specifically earns its cost
- You want **7B+** or **multi-GPU** training.
- You want **vLLM/SGLang-accelerated rollouts**. The article's recurring point is that
  rollouts dominate RL cost (Olmo 3 spends **5–14× more compute on inference** than on
  policy updates), so a fast sampler engine is the biggest single speedup.
- Long-running jobs where Colab's session limits and disconnects would kill you.

### The sharp connection back to the notebook
The moment you move to **RunPod + vLLM for fast rollouts, TIS [4] stops being decorative and
becomes load-bearing.** In this single-engine notebook the sampler *is* the learner, so the
importance ratio is ~1 and `use_tis` is a no-op. On a vLLM setup the sampler (vLLM, often
fp8/int8) and the learner (your FSDP/DeepSpeed engine) produce genuinely different token
probabilities — that off-policy gap is exactly what the truncated importance-sampling term in
`grpo_chunk_loss` corrects. So the cheap path (Colab, one engine) and the fast path (RunPod,
two engines) differ by precisely the one trick this notebook leaves switched off by default.

### Speed levers if you stay on a small GPU
`dynamic_sampling=False` (no oversampling), smaller `group_size`, shorter
`max_completion_len`, `use_lora=True`, larger `micro_bs` only if memory allows. Lower
`num_steps` for a smoke test — the curve *shape* is the lesson.